<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 6: </b>Embedding Models and Semantic Reasoning</h2>
<br>

In our previous notebook, we progressed our efforts towards the large document domain and learned how to apply our same techniques for dealing with a large base of relevant content! Along the way, we noted some challenges — specifically in the area of on-the-fly interpretation — that we still couldn't solve with our existing techniques. We will use this notebook to shift our focus to a different approach to help us approach this objective: **embedding models**.


<br>

### **Learning Objectives:**

- Familiarize yourself with embeddings — numerical representations of words, phrases, or documents — and how they enable deep learning models to process semantic meanings.

- Learn how to apply these embedding models in large-scale document processing, enhancing our existing methods of document summarization and knowledge extraction.

<br>

### **Questions To Think About:**

- How do embeddings contribute to a deeper understanding of document chunks, especially when dealing with inconsistencies or text conversion issues?

- How do we strike the right balance between detail and computational efficiency with embedding models? Are there ways to rephrase or canonicalize (standardize) your queries with LLMs? Alternatively, can we fine-tune them for specialized tasks?

- How can embedding models complement the running state chains and knowledge bases we've previously developed? *(Topic of next the notebook)*

<br>

### **Environment Setup:**

In [ ]:
## Necessary for Colab, not necessary for course environment
# %pip install -qq langchain langchain-nvidia-ai-endpoints fastembed gradio
# %pip install -qq arxiv pymupdf

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

----

<br>

## **Part 1:** Refreshing On Embedding Models

In this section, we'll review the ideas surrounding natural language processing with deep learning to define what embedding models are and how they're related to the tools we've taken for granted thus far.

<br>

### **Understanding Latent Embeddings**

Latent embeddings represent the middle ground in deep learning networks, bridging the gap between input and output. For instance, let's consider a lightweight two-layer network designed to classify [**MNIST digits**](https://en.wikipedia.org/wiki/MNIST_database). For this, the inputs and outputs might be flattened images and one-hot probability vectors, respectively. Then, the values produced by the first layer in this setup are the latent embeddings of the image which, by optimization, converge into useful representations for the final layer to use. This makes them **semantically rich embeddings** that may not be human-interpretable but can still be leveraged for their raw vector properties.

<br>

### **Word Embeddings: The Building Blocks of Language Models**

Word embeddings are high-dimensional vector representations of individual words, forming the backbone of deep language models. These embeddings are created through an optimization process within an end-to-end pipeline tailored for specific tasks. [**Word2vec**](https://en.wikipedia.org/wiki/Word2vec) is a prominent standalone example for those interested. In practical terms, a token from a language model's $v$-token vocabulary gets mapped from a token index to an $d$-dimensional token embedding:

$$\text{Token Index} \in \{0, 1, \cdots, v-1\} \to \text{Token Embedding} \in \mathbb{R}^{d}$$

For a sequence of $n$ tokens, this mapping extends to the entire sequence:

$$\text{Token Sequence} \in \{0, 1, \cdots, v-1\}^{v} \to \text{Embedding Sequence} \in \mathbb{R}^{n\times d}$$

<br>

### **Sentence/Document Embeddings: Capturing Context and Meaning**

When dealing with sentences or entire documents, embeddings play a crucial role in capturing context, meanings, and interactions between elements. Practically all large language models leverage a **transformer-like architecture** to generate these sentence/document embeddings. The transformer allows the network to communicate both per-token-level and sequence-level information as it becomes useful for the optimization problem.

<br>

### **Decoder Models in Language Generation**

Decoder models, commonly used in chatbots and other language generation tasks, start by taking a sequence of tokens as input. They embed these tokens into a latent sequence, applying unidirectional reasoning to focus on a specific part of the output sequence. From this focused, semantically dense point, the model predicts the next token in the sequence:

$$$$
$$\text{[ Next-Token Generation ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d}$$
$$(\text{Latent Sequence})[\text{last entry}] \in \mathbb{R}^{d} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$$$

This process continues, collapsing the token prediction from a vector to a realized token and building up a predicted sequence until a termination condition — such as a length limit or stop token — gets satisfied.

$$$$
$$\text{[ Autoregressive Generation ]}$$
$$(\text{Original + Predicted Embedding Sequence}) \in \mathbb{R}^{(n+1)\times e} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$\vdots$$
$$(\text{Original + Predicted Embedding Sequence}) \in \mathbb{R}^{(n+m)\times e} \to \text{Token Prediction} \in \mathbb{R}^{v}$$
$$$$
<br>

### **Encoder Models For Sequence Encoding**

Encoder models use a bidirectional architecture, making them suitable for different types of tasks compared to decoder models. They are particularly effective in tasks like token or sequence prediction. Letting $c$ be a number of classes or regressional values:

$$$$
$$\text{[ Per-Token Prediction ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d} \to \text{Per-Token Predictions} \in \mathbb{R}^{n\times c}$$

$$$$
$$\text{[ Full-Sequence Prediction ]}$$
$$\text{Embedding Sequence} \in \mathbb{R}^{n\times d} \to \text{Latent Sequence} \in \mathbb{R}^{n\times d}$$
$$(\text{Latent Sequence})[\text{0th entry}] \in \mathbb{R}^{d} \to \text{Sequence Prediction} \in \mathbb{R}^{c}$$

<br>

> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/encoder-decoder.png" width=1200px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1lhswkAgb5TlDxezg3qDNZQKbOMGFz7H5" width=1200px/> -->


----

<br>

## **Part 2:** Accessing the Course Embedding Model

For the rest of this notebook, we will use LangChain's `Embeddings` interface through the course model server. The server keeps the current CPU model loaded behind `/v1/embeddings`; the notebook still calls `embed_query` and `embed_documents`, while model loading and deployment stay on the server.

The stable `course/embedding` identifier does not tie the notebook to a local process or an API redirect. The `scope=all` catalog reports the current implementation when you want to inspect it. In this environment, MiniLM maps each input to a 384-dimensional vector and runs on the CPU.

In Colab, the commented alternative constructs the current model with FastEmbed. The embedding calls and retrieval exercises that follow remain the same.

<br>

### **Constructing Our Model**

Start by naming the service model used throughout the notebook, then construct its LangChain client.


In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embedding_model = "course/embedding"
embedding_model

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

embedder = NVIDIAEmbeddings(
    model=embedding_model,
    base_url="http://llm_client:9000/v1",
)

# In Colab, replace the service client above with:
# from langchain_community.embeddings import FastEmbedEmbeddings
# embedder = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

### **Embedding Queries and Documents Locally**

LangChain's `Embeddings` interface provides separate methods for embedding **queries** and **documents**. With MiniLM, both methods use the same encoder and return vectors in the same space. The method names reflect how those vectors are used: a query is embedded at search time, while documents are usually embedded in batches when the vector store is prepared.

<br>

#### **Query Embedding**
- **Purpose**: Designed for embedding shorter-form or question-like material, such as a simple statement or a question.
- **Method**: Utilizes `embed_query` for embedding each query individually.
- **Role in Retrieval**: Acts as the "key" creator to enable search (query process) in a document retrieval framework.
- **Usage Pattern**: Embedded dynamically, as needed, for comparison against a pre-processed collection of document embeddings.

<br>

#### **Document Embedding**
- **Purpose**: Tailored for longer-form or response-like content, including document chunks or paragraphs.
- **Method**: Employs `embed_documents` for batch processing of documents.
- **Role in Retrieval**: Acts as the "value" creator to make the searchable content for the retrieval system.
- **Usage Pattern**: Typically embedded en masse as a pre-processing step, creating a repository of document embeddings for future querying.

<br>

#### **Underlying Similarities and Practical Application**

Queries and documents must be embedded by the same model before we can compare them with cosine similarity. Use `embed_query` for an incoming search and `embed_documents` when preparing a document collection. If the embedding model changes, rebuild the document index before querying it again.

<br>

#### **Exploration with Example "Queries" and "Documents"**

Let's consider a set of example queries and documents to kickstart our exploration and understand these processes in action. These examples are carefully selected to highlight interesting properties and showcase our embedding model's capabilities for general text reasoning.


In [ ]:
# Example queries and documents
queries = [
    "What's the weather like in Rocky Mountains?",
    "What kinds of food is Italy known for?",
    "What's my name? I bet you don't remember...",
    "What's the point of life anyways?",
    "The point of life is to have fun :D"
]

documents = [
    "Komchatka's weather is cold, with long, severe winters.",
    "Italy is famous for pasta, pizza, gelato, and espresso.",
    "I can't recall personal names, only provide information.",
    "Life's purpose varies, often seen as personal fulfillment.",
    "Enjoying life's moments is indeed a wonderful approach.",
]

We can encode these passages via either the query or the document pathways. Since the method signatures differ due to their intended use cases, the syntax will be a bit different between the two options:


In [ ]:
%%time
# Embedding the queries
q_embeddings = [embedder.embed_query(query) for query in queries]

# Embedding the documents
d_embeddings = embedder.embed_documents(documents)

With our embeddings in hand, we can do a simple similarity check on the results to see which documents would have triggered as reasonable answers in a retrieval task. When your entries are ready, run the following code block to visualize the cross-similarity matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

def plot_cross_similarity_matrix(emb1, emb2):
    # Compute the similarity matrix between embeddings1 and embeddings2
    cross_similarity_matrix = cosine_similarity(np.array(emb1), np.array(emb2))

    # Plotting the cross-similarity matrix
    plt.imshow(cross_similarity_matrix, cmap='Greens', interpolation='nearest')
    plt.colorbar()
    plt.gca().invert_yaxis()
    plt.title("Cross-Similarity Matrix")
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_cross_similarity_matrix(q_embeddings, d_embeddings)
plt.xlabel("Query Embeddings")
plt.ylabel("Document Embeddings")
plt.show()

# queries = [
#     "What's the weather like in the Rocky Mountains?",
#     "What kinds of food is Italy known for?",
#     "What's my name? I bet you don't remember...",
#     "What's the point of life anyways?",
#     "The point of life is to have fun :D"]
# documents = [
#     "Komchatka's weather is cold, with long, severe winters.",
#     "Italy is famous for pasta, pizza, gelato, and espresso.",
#     "I can't recall personal names, only provide information.",
#     "Life's purpose varies, often seen as personal fulfillment.",
#     "Enjoying life's moments is indeed a wonderful approach."]

----

<br>

## **Part 3: [Exercise]** A Synthetic - But More Realistic - Example

The higher-scoring cells should correspond to the more plausible query/document pairs. In the next plot, we will also pass the documents through `embed_query` so that we can compare the two methods directly. Some embedding models apply different preprocessing to queries and documents. With a symmetric encoder such as MiniLM, the results may remain quite close:


In [ ]:
plt.figure(figsize=(8, 6))
plot_cross_similarity_matrix(
    q_embeddings,
    [embedder.embed_query(doc) for doc in documents]
)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Query Embeddings (of documents)")
plt.show()

More generally, a bi-encoder lets us embed documents once and compare those stored vectors with each new query. Models with distinct query and document pathways can also optimize each pathway for a different input form. MiniLM uses a symmetric encoder, so this example focuses on the retrieval workflow rather than on differences between two separately trained encoders. To see how the input itself affects the comparison, we can flesh out our documents into longer-form variations and try the experiment again.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from pprint import pprint

expound_prompt = ChatPromptTemplate.from_template(
    "Generate part of a longer story that could reasonably answer all"
    " of these questions somewhere in its contents: {questions}\n"
    " Make sure the passage only answers the following concretely: {q1}."
    " Give it some weird formatting, and try not to answer the others."
    " Do not include any commentary like 'Here is your response'"
)

###############################################################################################
## BEGIN TODO

expound_chain = (
    ## TODO: flesh out documents into a more verbose form by implementing the expound_chain 
    ##  which takes advantage of the prompt and llm provided above.
    {}
)

longer_docs = []
for i, q in enumerate(queries):
    ## TODO: Invoke the expound_chain pipeline as appropriate
    longer_doc = ""
    pprint(f"\n\n[Query {i+1}]")
    print(q)
    pprint(f"\n\n[Document {i+1}]")
    print(longer_doc)
    pprint("-"*64)
    longer_docs += [longer_doc]

## END TODO
###############################################################################################

-----

When you're happy with the longer-form documents, run the code below to compare their embeddings with the original queries. The scores may shift because the rewritten passages introduce new words and structure, even though MiniLM uses the same encoder for both methods.

For other embedding models, follow the documented query and document methods because those pathways may apply different preprocessing or training objectives. Whichever model you choose, use it consistently for both the stored document vectors and the queries compared against them.


In [ ]:
## Bound this demonstration to 2048 characters; use a token-aware splitter for production inputs.
longer_docs_cut = [doc[:2048] for doc in longer_docs]

q_long_embs = [embedder.embed_query(doc) for doc in longer_docs_cut]
d_long_embs = embedder.embed_documents(longer_docs_cut)

## The difference for any particular example may be very small.
## We've raised the similarity matrix to the power of 5 to try and spot a difference.
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plot_cross_similarity_matrix(q_embeddings, q_long_embs)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Query Embeddings (of long documents)")

plt.subplot(1, 2, 2)
plot_cross_similarity_matrix(q_embeddings, d_long_embs)
plt.xlabel("Query Embeddings (of queries)")
plt.ylabel("Document Embeddings (of long documents)")
plt.show()

<br>

**NOTE:** To see two drastically different bi-encoder components, consider checking out the [Contrastive Language-Image Pre-training model (CLIP)](https://openai.com/index/clip/). This bi-encoder pair synergizes over a much wider modality gap to connect the image and text modalities instead of the query and document ones.


----

<br>

## **Part 4: [Exercise]** Embeddings For Semantic Guardrails

In the next notebook, we will start to use higher-level utilities which will take our embedding model and use it under the hood. With that being said, there are several important concepts that can still be explored while the raw methods are still fresh!

Specifically, we can use it as a backbone for a critical component of productionalized models: **semantic guardrailing**. Specifically, we can use the embeddings to filter out messages that are unlikely to be useful (or are actively harmful) for our chatbot to answer!

**This exercise has been segmented into [**`64_guardrails.ipynb`**](64_guardrails.ipynb).**

-----

## **Part 5:** Wrap-Up

By the end of this notebook, you should be familiar with the value proposition of a semantic embedding model and understand how we can use it to search through a dataset for relevant information!

### <font color="#76b900">**Great Job!**</font>

### **Next Steps:**
1. **[Optional]** Revisit the **"Questions To Think About" Section** at the top of the notebook and think about some possible answers.
2. **[Advanced]** If you have the time, please check out and try to complete the **Notebook 6.4**, which covers semantic guardrails.
3. Continue to the next video, which will discuss **Retrieval with Vectorstores**.
4. After the video, go on to the corresponding notebook on **Retrieval with Vectorstores**.

---

<div style="width: 55%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>